In [1]:
from google.colab import drive
drive.mount("/content/drive")

!pip -q install -U "transformers>=4.46.0" "peft>=0.13.0" "accelerate>=1.0.0" "torchao>=0.16.0" scikit-learn safetensors tqdm

import gc, json, math, os, random, shutil
import numpy as np
import torch
from peft import LoraConfig, PeftConfig, PeftModel, TaskType, get_peft_model
from safetensors.torch import load_file
from sklearn.metrics import accuracy_score, average_precision_score, balanced_accuracy_score, confusion_matrix, matthews_corrcoef, precision_recall_fscore_support
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, get_linear_schedule_with_warmup
import torch.nn.functional as F

os.environ["TOKENIZERS_PARALLELISM"] = "false"
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

model_id = "EleutherAI/pythia-1.4b-deduped"
root = "/content/drive/MyDrive/Test"
data_dir = os.path.join(root, "PreparedData")
en_dir = os.path.join(root, "English")
out_dir = os.path.join(root, "FinalCompare", "final_compare_results")
en_out = os.path.join(en_dir, "english_results")

os.makedirs(out_dir, exist_ok=True)
os.makedirs(en_out, exist_ok=True)

seed_value = 42
device = "cuda"
dtype = torch.float16
amp = True
train_bs = 16
eval_bs = 16
accum = 1
lr = 5e-5
wd = 0.01
warmup = 0.05
save_mistakes = True

answers = (" False", " True")
modules = ["query_key_value", "dense", "dense_h_to_4h", "dense_4h_to_h"]
lora_cfg = dict(r=64, lora_alpha=64, lora_dropout=0.10)

lams = [0.0, 0.05, 0.1, 0.2, 0.3, 0.5, 0.75, 1.0]

en_best = os.path.join(en_dir, "adapter_task_en_best")
en_last = os.path.join(en_dir, "adapter_task_en_last")

files = {
    "en_main": os.path.join(data_dir, "english_train_main.jsonl"),
    "en_finish": os.path.join(data_dir, "english_train_finish.jsonl"),
    "en_val": os.path.join(data_dir, "english_val.jsonl"),
    "en_test": os.path.join(data_dir, "english_test.jsonl"),
    "py_val": os.path.join(data_dir, "python_val.jsonl"),
    "py_test": os.path.join(data_dir, "python_test.jsonl"),
    "summary": os.path.join(out_dir, "phase3_summary.json"),
    "rows": os.path.join(out_dir, "phase3_best_rows.jsonl"),
    "mistakes": os.path.join(out_dir, "phase3_test_mistakes.jsonl"),
    "en_summary": os.path.join(en_out, "english_run_summary.json"),
    "en_mistakes": os.path.join(en_out, "english_test_mistakes.jsonl"),
}

aux_paths = {
    "atoms": os.path.join(root, "Python", "python_atoms_aux", "best"),
    "codeparrot": os.path.join(root, "Python", "codeparrot_python_aux", "best"),

    "bridge_baseline": os.path.join(root, "Python", "test_bridge_simple", "baseline", "best"),
    "bridge_full_prompts_50": os.path.join(root, "Python", "test_bridge_simple", "full_prompts_50", "best"),
    "bridge_format_2x": os.path.join(root, "Python", "test_bridge_simple", "format_2x", "best"),
    "bridge_full_prompts_50_format_2x": os.path.join(root, "Python", "test_bridge_simple", "full_prompts_50_format_2x", "best"),
    "bridge_contrastive_0p5x": os.path.join(root, "Python", "test_bridge_simple", "contrastive_0p5x", "best"),
    "bridge_contrastive_2x": os.path.join(root, "Python", "test_bridge_simple", "contrastive_2x", "best"),
    "bridge_mean_pooling": os.path.join(root, "Python", "test_bridge_simple", "mean_pooling", "best"),
}

def set_seed(x=seed_value):
    random.seed(x)
    np.random.seed(x)
    torch.manual_seed(x)
    torch.cuda.manual_seed_all(x)

def clear():
    gc.collect()
    torch.cuda.empty_cache()

def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(x) for x in f if x.strip()]

def write_json(path, x):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(x, f, indent=2, ensure_ascii=False)

def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

def ids(rows):
    return {r["uid"] for r in rows}

def split_report(en_main, en_finish, en_val, en_test, py_val, py_test):
    train = ids(en_main) | ids(en_finish)
    val = ids(en_val) | ids(py_val)
    test = ids(en_test) | ids(py_test)
    overlap = {
        "train_val": len(train & val),
        "train_test": len(train & test),
        "val_test": len(val & test),
    }
    assert overlap["train_val"] == 0
    assert overlap["train_test"] == 0
    assert overlap["val_test"] == 0
    return overlap

def pos_rate(rows):
    return float(np.mean([int(r["label"]) for r in rows]))

def fmt(x):
    return f"{float(x):.3f}".rstrip("0").rstrip(".") or "0"

def show(name, m):
    return f"{name}: loss={m['loss']:.4f} | acc={m['accuracy']:.3f} | bal_acc={m['balanced_accuracy']:.3f} | f1={m['f1_pos']:.3f} | mcc={m['mcc']:.3f} | fp={m['fp']} fn={m['fn']}"

def key(m):
    return m["mcc"]

def slim(m):
    return {k: v for k, v in m.items() if k != "mistakes"}

def init_tok():
    set_seed()
    tok = AutoTokenizer.from_pretrained(model_id)
    tok.pad_token = tok.eos_token
    lens = [len(tok(x, add_special_tokens=False).input_ids) for x in answers]
    ids_ = [tok(x, add_special_tokens=False).input_ids[0] for x in answers]
    print(f"[eval] candidate token lengths | neg={lens[0]} | pos={lens[1]}")
    return tok, ids_

class TextData(Dataset):
    def __init__(self, rows, tok):
        self.rows = []
        for r in rows:
            x = tok(r["text"], add_special_tokens=False).input_ids
            self.rows.append({"input_ids": x, "label": int(r["label"])})

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        return self.rows[i]


def batch_pad(batch, pad_id):
    n = max(len(x["input_ids"]) for x in batch)
    out = {"input_ids": [], "attention_mask": [], "label": []}
    for x in batch:
        k = n - len(x["input_ids"])
        out["input_ids"].append(x["input_ids"] + [pad_id] * k)
        out["attention_mask"].append([1] * len(x["input_ids"]) + [0] * k)
        out["label"].append(x["label"])
    return {k: torch.tensor(v) for k, v in out.items()}


def loader(rows, tok):
    return DataLoader(
        TextData(rows, tok),
        batch_size=train_bs,
        shuffle=True,
        pin_memory=True,
        collate_fn=lambda b: batch_pad(b, tok.pad_token_id),
    )

def make_base():
    return AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=dtype, low_cpu_mem_usage=True).to(device).eval()

def make_lora():
    m = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=dtype, low_cpu_mem_usage=True)
    m.config.use_cache = False
    m.gradient_checkpointing_enable()
    m.enable_input_require_grads()
    cfg = LoraConfig(task_type=TaskType.CAUSAL_LM, inference_mode=False, bias="none", target_modules=modules, **lora_cfg)
    return get_peft_model(m, cfg).to(device)

def make_peft(path):
    m = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=dtype, low_cpu_mem_usage=True)
    return PeftModel.from_pretrained(m, path).to(device).eval()

def save_model(m, tok, path):
    if os.path.isdir(path):
        shutil.rmtree(path)
    os.makedirs(path, exist_ok=True)
    m.save_pretrained(path, safe_serialization=True)
    tok.save_pretrained(path)

def metrics(gold, prob, pred):
    p, r, f1, _ = precision_recall_fscore_support(gold, pred, average="binary", pos_label=1, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(gold, pred, labels=[0, 1]).ravel()
    return {
        "accuracy": float(accuracy_score(gold, pred)),
        "balanced_accuracy": float(balanced_accuracy_score(gold, pred)),
        "precision_pos": float(p),
        "recall_pos": float(r),
        "f1_pos": float(f1),
        "mcc": float(matthews_corrcoef(gold, pred)) if len(np.unique(pred)) > 1 else 0.0,
        "average_precision": float(average_precision_score(gold, prob)) if len(np.unique(gold)) > 1 else 0.0,
        "tp": int(tp),
        "fp": int(fp),
        "tn": int(tn),
        "fn": int(fn),
        "positive_rate_gold": float(np.mean(gold)),
        "positive_rate_pred": float(np.mean(pred)),
    }

def evaluate(m, tok, rows, ids_, keep=False):
    false_id, true_id = ids_
    gold = np.array([int(r["label"]) for r in rows])
    pred, prob = [], []
    loss_sum, count = 0.0, 0
    m.eval()

    for i in tqdm(range(0, len(rows), eval_bs), leave=False):
        part = rows[i:i + eval_bs]
        enc = tok([r["text"] for r in part], return_tensors="pt", padding=True, add_special_tokens=False)
        enc = {k: v.to(device) for k, v in enc.items()}
        labels = torch.tensor([int(r["label"]) for r in part], device=device)

        with torch.inference_mode(), torch.amp.autocast("cuda", enabled=amp):
            z = m(**enc).logits

        last = enc["attention_mask"].sum(1) - 1
        z = z[torch.arange(z.size(0), device=device), last][:, [false_id, true_id]].float()

        loss_sum += F.cross_entropy(z, labels, reduction="sum").item()
        count += labels.numel()

        pred += (z[:, 1] > z[:, 0]).long().cpu().tolist()
        prob += torch.softmax(z, -1)[:, 1].cpu().tolist()

    pred = np.array(pred)
    prob = np.array(prob)

    out = metrics(gold, prob, pred)
    out["loss"] = float(loss_sum / count)
    out["evaluated_rows"] = len(rows)
    out["mistake_count"] = int(np.sum(pred != gold))
    out["mistake_rate"] = float(np.mean(pred != gold))

    if keep:
        out["mistakes"] = [
            {
                "row_index": i,
                "gold_label": int(gold[i]),
                "pred_label": int(pred[i]),
                "prob_true": float(prob[i]),
                "prob_false": float(1 - prob[i]),
                **{k: v for k, v in rows[i].items() if k != "label"},
            }
            for i in range(len(rows)) if pred[i] != gold[i]
        ]

    return out

def train_english(tok, ids_):
    en_main = read_jsonl(files["en_main"])
    en_finish = read_jsonl(files["en_finish"])
    en_val = read_jsonl(files["en_val"])
    en_test = read_jsonl(files["en_test"])
    py_val = read_jsonl(files["py_val"])
    py_test = read_jsonl(files["py_test"])
    overlap = split_report(en_main, en_finish, en_val, en_test, py_val, py_test)

    phases = [("main", en_main, 1), ("finish", en_finish, 1)]
    m = make_lora()
    dls = [(name, loader(rows, tok), epochs) for name, rows, epochs in phases]
    steps = sum(math.ceil(len(dl) / accum) * epochs for _, dl, epochs in dls)
    opt = torch.optim.AdamW((p for p in m.parameters() if p.requires_grad), lr=lr, weight_decay=wd)
    sch = get_linear_schedule_with_warmup(opt, int(steps * warmup), steps)
    scaler = torch.amp.GradScaler("cuda", enabled=amp)

    hist, best, update = [], None, 0
    for name, dl, epochs in dls:
        for ep in range(1, epochs + 1):
            m.train()
            opt.zero_grad(set_to_none=True)
            total, count = 0.0, 0
            for step, batch in enumerate(tqdm(dl, desc=f"{name} {ep}/{epochs}"), 1):
                batch = {k: v.to(device) for k, v in batch.items()}
                labels = batch.pop("label").to(device)

                with torch.amp.autocast("cuda", enabled=amp):
                    z = m(**batch).logits
                    last = batch["attention_mask"].sum(1) - 1
                    z = z[torch.arange(z.size(0), device=device), last][:, ids_].float()
                    loss = F.cross_entropy(z, labels) / accum
                scaler.scale(loss).backward()
                if step % accum == 0 or step == len(dl):
                    scaler.step(opt)
                    scaler.update()
                    opt.zero_grad(set_to_none=True)
                    sch.step()
                    update += 1
                bsz = batch["input_ids"].size(0)
                total += loss.detach().float().item() * accum * bsz
                count += bsz

            val_m = evaluate(m, tok, en_val, ids_)
            hist.append({"phase": name, "epoch": ep, "update": update, "train_loss": total / count, "val": slim(val_m)})
            print(show(f"english/{name}/{ep}", val_m))
            if best is None or key(val_m) > key(best):
                best = val_m
                save_model(m, tok, en_best)
                print("saved:", en_best)

    save_model(m, tok, en_last)
    del m
    clear()

    m = make_peft(en_best)
    best_val = evaluate(m, tok, en_val, ids_)
    best_test = evaluate(m, tok, en_test, ids_, save_mistakes)
    del m
    clear()

    summary = {
        "model_id": model_id,
        "settings": {
            "seed": seed_value,
            "dtype": str(dtype),
            "train_bs": train_bs,
            "eval_bs": eval_bs,
            "grad_accum": accum,
            "lr": lr,
            "weight_decay": wd,
            "warmup": warmup,
            "lora": lora_cfg,
            "english_epochs": {"main": 1, "finish": 1},
        },
        "paths": {"best": en_best, "last": en_last, "results": en_out},
        "counts": {"train_main": len(en_main), "train_finish": len(en_finish), "english_val": len(en_val), "english_test": len(en_test), "python_val": len(py_val), "python_test": len(py_test)},
        "positive_rates": {"train_main": pos_rate(en_main), "train_finish": pos_rate(en_finish), "english_val": pos_rate(en_val), "english_test": pos_rate(en_test), "python_val": pos_rate(py_val), "python_test": pos_rate(py_test)},
        "uid_overlap": overlap,
        "history": hist,
        "best_val_during_train": slim(best),
        "english_best": {"english_val": slim(best_val), "english_test": slim(best_test)},
    }
    write_json(files["en_summary"], summary)
    if save_mistakes:
        write_jsonl(files["en_mistakes"], [{"eval_name": "english_best", "split": "english_test", **r} for r in best_test.get("mistakes", [])])
    print(show("english_best/val", best_val))
    print(show("english_best/test", best_test))
    return summary

def clean_name(k):
    for p in ("base_model.model.", "base_model.", "model."):
        if k.startswith(p):
            return k[len(p):]
    return k

def get_part(state, p, name):
    return state[p + f".{name}.weight"] if p + f".{name}.weight" in state else state[p + f".{name}.default.weight"]

def load_delta(path):
    cfg = PeftConfig.from_pretrained(path)
    state = load_file(os.path.join(path, "adapter_model.safetensors"))
    scale = cfg.lora_alpha / cfg.r
    out = {}
    for p in sorted({k.split(".lora_A")[0] for k in state if ".lora_A" in k}):
        a = get_part(state, p, "lora_A").float()
        b = get_part(state, p, "lora_B").float()
        d = (b @ a) * scale
        if getattr(cfg, "fan_in_fan_out", False):
            d = d.T
        out[clean_name(p) + ".weight"] = d.cpu().contiguous()
    return out

def add_delta(m, d, c=1.0):
    s = m.state_dict()
    with torch.no_grad():
        for k, v in d.items():
            s[k].add_(v.to(s[k].device, s[k].dtype), alpha=float(c))

def run_eval(tok, ids_, val, test, parts):
    m = make_base()
    for c, d in parts:
        add_delta(m, d, c)
    out = {"python_val": evaluate(m, tok, val, ids_), "python_test": evaluate(m, tok, test, ids_, save_mistakes)}
    del m
    clear()
    return out

def run_sweep(tok, ids_, val, test, en, aux):
    m = make_base()
    add_delta(m, en, 1.0)
    cur, best_lam, best_val, sweep = 0.0, None, None, {}
    for lam in lams:
        add_delta(m, aux, float(lam) - cur)
        cur = float(lam)
        val_m = evaluate(m, tok, val, ids_)
        sweep[fmt(lam)] = slim(val_m)
        print("  ", fmt(lam), show("val", val_m))
        if best_val is None or key(val_m) > key(best_val):
            best_lam, best_val = float(lam), val_m
    add_delta(m, aux, best_lam - cur)
    out = {"python_val": evaluate(m, tok, val, ids_), "python_test": evaluate(m, tok, test, ids_, save_mistakes)}
    del m
    clear()
    return best_lam, sweep, out

def rows(summary):
    out = []
    for split in ["python_val", "python_test"]:
        out.append({"name": "english_only", "kind": "english_only", "lambda": None, "split": split, **summary["english_only"][split]})
    for name, x in summary["aux"].items():
        for kind in ["aux_only", "english_plus_aux"]:
            for split in ["python_val", "python_test"]:
                out.append({"name": name, "kind": kind, "lambda": x[kind]["lambda"], "split": split, **x[kind][split]})
    return out


Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 114.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 114.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 131.3 MB/s eta 0:00:00


In [2]:
tok, ids_ = init_tok()
english_summary = train_english(tok, ids_)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/396 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

...

english/finish/1: loss=0.2658 | acc=0.920 | bal_acc=0.803 | f1=0.590 | mcc=0.550 | fp=124 fn=73
saved: /content/drive/MyDrive/Test/English/adapter_task_en_best


Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

  0%|          | 0/154 [00:00<?, ?it/s]

  0%|          | 0/172 [00:00<?, ?it/s]

english_best/val: loss=0.2658 | acc=0.920 | bal_acc=0.803 | f1=0.590 | mcc=0.550 | fp=124 fn=73
english_best/test: loss=0.2281 | acc=0.923 | bal_acc=0.835 | f1=0.662 | mcc=0.622 | fp=133 fn=79


In [3]:
tok, ids_ = init_tok()
py_val = read_jsonl(files["py_val"])
py_test = read_jsonl(files["py_test"])

print("python_val rows:", len(py_val), "positive_rate:", pos_rate(py_val))
print("python_test rows:", len(py_test), "positive_rate:", pos_rate(py_test))

en = load_delta(en_best)
summary = {
    "model_id": model_id,
    "settings": {
        "seed": seed_value,
        "dtype": str(dtype),
        "eval_bs": eval_bs,
        "lambda_list": lams,
        "selection_metric": "highest validation MCC",
    },
    "paths": {"english": en_best, "aux": aux_paths, "results": out_dir},
    "counts": {"python_val": len(py_val), "python_test": len(py_test)},
    "positive_rates": {"python_val": pos_rate(py_val), "python_test": pos_rate(py_test)},
    "english_only": None,
    "aux": {},
}

print("\nenglish_only")
res = run_eval(tok, ids_, py_val, py_test, [(1.0, en)])
summary["english_only"] = {k: slim(v) for k, v in res.items()}
print(show("english_only/val", res["python_val"]))
print(show("english_only/test", res["python_test"]))

mistakes = [{"eval_name": "english_only", "split": "python_test", **r} for r in res["python_test"].get("mistakes", [])]
write_json(files["summary"], summary)

for name, path in aux_paths.items():
    print("\n" + name)
    aux = load_delta(path)

    aux_only = run_eval(tok, ids_, py_val, py_test, [(1.0, aux)])
    print(show("aux_only/val", aux_only["python_val"]))
    print(show("aux_only/test", aux_only["python_test"]))

    best_lam, sweep, best = run_sweep(tok, ids_, py_val, py_test, en, aux)
    print("best_lambda:", fmt(best_lam))
    print(show("english_plus_aux/val", best["python_val"]))
    print(show("english_plus_aux/test", best["python_test"]))

    summary["aux"][name] = {
        "path": path,
        "aux_only": {
            "lambda": 1.0,
            "python_val": slim(aux_only["python_val"]),
            "python_test": slim(aux_only["python_test"]),
        },
        "english_plus_aux": {
            "lambda": best_lam,
            "val_sweep": sweep,
            "python_val": slim(best["python_val"]),
            "python_test": slim(best["python_test"]),
        },
    }

    mistakes += [{"eval_name": f"{name}/aux_only", "split": "python_test", **r} for r in aux_only["python_test"].get("mistakes", [])]
    mistakes += [{"eval_name": f"{name}/english_plus_aux", "split": "python_test", **r} for r in best["python_test"].get("mistakes", [])]

    write_json(files["summary"], summary)
    write_jsonl(files["rows"], rows(summary))
    if save_mistakes:
        write_jsonl(files["mistakes"], mistakes)

write_json(files["summary"], summary)
write_jsonl(files["rows"], rows(summary))
if save_mistakes:
    write_jsonl(files["mistakes"], mistakes)

print("\nsaved:")
print(files["summary"])
print(files["rows"])
if save_mistakes:
    print(files["mistakes"])


[eval] candidate token lengths | neg=1 | pos=1
python_val rows: 2452 positive_rate: 0.08768352365415986
python_test rows: 2748 positive_rate: 0.1044395924308588

english_only


Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

  0%|          | 0/154 [00:00<?, ?it/s]

  0%|          | 0/172 [00:00<?, ?it/s]

english_only/val: loss=0.7609 | acc=0.453 | bal_acc=0.505 | f1=0.154 | mcc=0.005 | fp=1248 fn=93
english_only/test: loss=0.7416 | acc=0.490 | bal_acc=0.515 | f1=0.183 | mcc=0.019 | fp=1271 fn=130

atoms


...

  0%|          | 0/154 [00:00<?, ?it/s]

   1 val: loss=0.4063 | acc=0.904 | bal_acc=0.508 | f1=0.049 | mcc=0.041 | fp=26 fn=209


  0%|          | 0/154 [00:00<?, ?it/s]

  0%|          | 0/172 [00:00<?, ?it/s]

best_lambda: 1
english_plus_aux/val: loss=0.4063 | acc=0.904 | bal_acc=0.508 | f1=0.049 | mcc=0.041 | fp=26 fn=209
english_plus_aux/test: loss=0.4255 | acc=0.883 | bal_acc=0.493 | f1=0.000 | mcc=-0.039 | fp=35 fn=287

saved:
/content/drive/MyDrive/Test/FinalCompare/final_compare_results/phase3_summary.json
/content/drive/MyDrive/Test/FinalCompare/final_compare_results/phase3_best_rows.jsonl
/content/drive/MyDrive/Test/FinalCompare/final_compare_results/phase3_test_mistakes.jsonl
